<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQk4anYFFsY3OTqb6RP4oTBhqPM0P1u80tSRA&usqp=CAU" width=75 height=50>

*Algirdas Laukaitis* — originalus Colab CNN užrašų knygelės autorius.

Ši versija: **1.4 užduotis** (Gilusis mokymas, VILNIUS TECH). Originalas pakartotas ląstelė po ląstelės, bet CIFAR-10 pakeistas **tf_flowers** duomenų rinkiniu.

<br><br>

---



# PART 1

Trumpai: 1 dalyje paruošiame vaizdus, sudarome konvoliucinį tinklą (CNN), jį apmokome ir įvertiname tikslumą.

<font color='SeaGreen'>Download and prepare the tf_flowers dataset</font>        <img src="https://storage.googleapis.com/tfds-data/visualization/fig/tf_flowers-3.0.1.png" width=100 height=100>

**Kodėl ne CIFAR-10?** Dėstytojo Colab naudoja CIFAR-10 (32×32 RGB, 10 klasių: lėktuvas, automobilis, katė ir t. t.). Užduotis reikalauja **kitų** viešai atsisiunčiamų vaizdų. Pasirinktas `tf_flowers`: 5 gėlių rūšys, RGB nuotraukos, ~3670 vaizdų, archyvas iš Google Storage be Kaggle prisijungimo. Vaizdai sumažinami iki 32×32, kad CNN architektūra liktų tokia pati kaip originalo, o 2 dalies RGB filtrų vizualizacija veiktų be pakeitimų.

In [ ]:
#@title Kodas, kuris įkelia tf_flowers rinkinį — RGB gėlių nuotraukas (5 klasės).

import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np

# Originaliame Colab: class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
# Originaliame Colab Keras CIFAR-10 atsisiuntimas buvo praleistas (lėtas), o duomenys krauti iš Hugging Face.
# Čia Keras load_data() netinka — tf_flowers nėra Keras bundled rinkinys.
class_names = ['daisy', 'dandelion', 'roses', 'sunflowers', 'tulips']


**Kaip veikia duomenų įkėlimas.** Originaliame dokumente 6 ląstelė darė `load_dataset("uoft-cs/cifar10")` ir sudėjo `img` / `label` į `numpy` masyvus. Čia tas pats galutinis formatas (`train_images`, `train_labels`, `test_images`, `test_labels`), tik šaltinis kitas: viešas TensorFlow `flower_photos.tgz` archyvas. Klasės imamos iš aplankų vardų, vaizdai keičiami į 32×32, tada 80/20 treniravimo/testo dalijimas (fiksuotas `seed=42`).

In [ ]:
import pathlib
from PIL import Image

dataset_url = "https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz"
archive = tf.keras.utils.get_file(
    fname="flower_photos.tgz",
    origin=dataset_url,
    extract=True,
)
archive_path = pathlib.Path(archive)
candidates = [
    archive_path.parent / "flower_photos",
    archive_path.with_suffix(""),
    archive_path.parent / "flower_photos.tgz_extracted" / "flower_photos",
]
data_dir = next(p for p in candidates if p.is_dir() and (p / "daisy").is_dir())
print("data_dir:", data_dir)

img_size = 32
images = []
labels = []
for label, class_name in enumerate(class_names):
    class_dir = data_dir / class_name
    for path in sorted(class_dir.glob("*.jpg")):
        img = Image.open(path).convert("RGB").resize((img_size, img_size), Image.Resampling.BILINEAR)
        images.append(np.array(img, dtype=np.uint8))
        labels.append(label)

images = np.stack(images)
labels = np.array(labels, dtype=np.int64).reshape(-1, 1)

rng = np.random.default_rng(42)
idx = rng.permutation(len(images))
split = int(0.8 * len(images))
train_idx, test_idx = idx[:split], idx[split:]
train_images, test_images = images[train_idx], images[test_idx]
train_labels, test_labels = labels[train_idx], labels[test_idx]

print("train:", train_images.shape, train_labels.shape)
print("test:", test_images.shape, test_labels.shape)
print("counts:", {n: int((labels[:, 0] == i).sum()) for i, n in enumerate(class_names)})


In [ ]:
#@title Normalizing the pixel values is crucial, ensuring our data is ready for the challenges ahead

# Pixeliai 0..255 pakeičiami į 0..1 — taip Adam optimizatorius ir ReLU sluoksniai stabiliau mokosi.
train_images, test_images = train_images / 255.0, test_images / 255.0


<br><br>

---
###<font color='SeaGreen'>Let's plot the first 100 images </font>

Pirmi 100 treniravimo vaizdų — greita patikra, kad etiketės ir RGB kanalai sutampa su tuo, ką matome.

In [ ]:
#@title Peržiūrime šimtą tf_flowers vaizdų, kad įsitikintume, jog duomenys sveiki.
plt.figure(figsize=(15,15))
for i in range(100):
    plt.subplot(10,10,i+1)
    plt.xticks([])
    plt.yticks([])
    plt.grid(False)
    plt.imshow(train_images[i])
    plt.xlabel(class_names[train_labels[i][0]])
plt.show()




<br><br><br><br>


# <font color='SeaGreen'>Convolutional Neural Network</font>        <img src="https://upload.wikimedia.org/wikipedia/commons/6/63/Typical_cnn.png" width=500 height=150>

**Kaip veikia CNN.** Konvoliucinis sluoksnis slankioja mažu filtru (čia 3×3) per vaizdą ir skaičiuoja skaliarinę sandaugą — taip randami lokalūs šablonai (kraštinės, spalvos, tekstūros). `MaxPooling2D(2,2)` sumažina erdvinį dydį perpus, palikdamas stipriausią signalą. Po kelių konvoliucijų `Flatten` paverčia žemėlapį vektoriumi, o `Dense` sluoksniai priima klasės sprendimą. Originalo išėjimas turėjo 10 neuronų (CIFAR-10); čia **5**, nes tf_flowers turi 5 klases. Visa kita architektūra palikta identiška.

<br><br>

---
###<font color='SeaGreen'>Step 2. Model </font>


In [ ]:
model = models.Sequential()

model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32, 32, 3)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(5))
'''
model.add(layers.Flatten(input_shape=(32, 32, 3)))
model.add(layers.Dense(5))
'''


In [ ]:

model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  metrics=['accuracy'])

model.summary() # Number of parameters per filter: (filter size * input channels) + 1 (bias)  ; Total parameters: number of filters * parameters per filter


In [ ]:
from tensorflow.keras.utils import plot_model
plot_model(model, to_file='model.png', show_shapes=True, show_layer_names=True)


<br><br>

---
###<font color='SeaGreen'>Step 3. Train and evaluate the model</font>

`model.fit` kelis kartus (epochas) praleidžia visą treniravimo imtį. `validation_data` po kiekvienos epochos matuoja testinę imtį — taip matome, ar tinklas mokosi, ar tik įsimena. Originalas: **20** CNN epochų. CPU aplinkoje sumažinta iki **8**; to pakanka palyginti CNN su perceptronu. Nuostolis `SparseCategoricalCrossentropy(from_logits=True)`, nes paskutinis sluoksnis be softmax (grąžina logit'us).

In [ ]:
# Originaliame Colab: epochs=20. Sumažinta CPU laikui.
history = model.fit(train_images, train_labels, epochs=8, validation_data=(test_images, test_labels))


####<font color='SeaGreen'>Evaluate the model</font>


In [ ]:
#@title CNN kreivės ir testinis tikslumas. Originaliame CIFAR-10 Colab minėta ~70 %; tf_flowers (5 klasės, 32×32) rezultatas bus kitoks.

plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.1, 1])
plt.legend(loc='lower right')

test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)
print(test_acc)
cnn_test_acc = test_acc



<br><br>

---
# PART 2

2 dalis nebe moko tinklo — ji **paaiškina**, ką jau išmoko pirmasis konvoliucinis sluoksnis: piešiami filtrų svoriai ir požymių žemėlapiai (feature maps). Kodas paliktas kaip dėstytojo, nes vaizdai vis dar 32×32×3.

<font color='SeaGreen'>Visualizing Filters and Feature Maps</font> <img src="https://upload.wikimedia.org/wikipedia/commons/1/19/2D_Convolution_Animation.gif" width=200 height=200>


The simplest way to visualize learned filters in a neural network is to plot them as two-dimensional images. These filters, essentially weights, have a spatial relationship due to their two-dimensional structure. To view these filters, you can access the model's layers and identify convolutional layers with 'conv' in their names. Each convolutional layer has filter weights and bias values, which you can retrieve and examine. Typically, convolutional layers use 3x3 filters, and it's essential that the filter depth matches the input's channel depth. This process helps you understand the model's filter shapes and their compatibility with input channels.

**Lietuviškai.** Pirmasis `Conv2D` turi 32 filtrus formos 3×3×3 (aukštis × plotis × RGB kanalai). `get_weights()` grąžina filtrus ir poslinkius (bias). Piešdami atskirus kanalus matome, į kokį spalvinį šabloną reaguoja filtras. Toliau tas pats filtras praleidžiamas per vieną nuotrauką — gaunamas požymių žemėlapis. Galiausiai konvoliucija apskaičiuojama ranka (`numpy`) ir palyginama su Keras rezultatu (po ReLU).

In [ ]:
#@title These are the building blocks of our neural networks.

filters, biases = model.layers[0].get_weights()

ix = 1
for filter_i in range(2):
	filter_tmp = filters[:, :, :, filter_i]
	for channel_j in range(3):  # plot each channel separately
		ax = plt.subplot(2, 3, ix)
		ax.set_xticks([])
		ax.set_yticks([])
		plt.imshow(filter_tmp[:, :, channel_j], cmap='gray')
		ix += 1
plt.show()


In [ ]:
filter_tmp[:, :, 0]


In [ ]:
#@title 📤 Let's delve into filter outputs.

model_layers_0 = tf.keras.Model(inputs=model.inputs, outputs=model.layers[0].output)  # redefine model to output right after the first hidden layer


In [ ]:

img = train_images[1]  # load the image
img = np.expand_dims(img, axis=0)  # expand dimensions so that it represents a single 'sample'


feature_maps = model_layers_0.predict(img)  # get feature map for first hidden layer

#plt.figure(figsize=(5, 5));plt.imshow(train_images[1]);plt.axis('off');plt.show()


In [ ]:
transposed_img = np.transpose(train_images[1] , (2, 0, 1))
np.set_printoptions(threshold=np.inf)
for row in transposed_img[0]:
    formatted_row = ["{:0.2f}".format(element) for element in row]
    print(" ".join(formatted_row))


In [ ]:
#@title 📍 feature maps created by the filters from the first layer of our trained CNN. Each filter extracts a unique facet.

ix = 1
for _ in range(4):
  for _ in range(4):
    if ix > 32:
      break
    # specify subplot and turn of axis
    ax = plt.subplot(4, 4, ix)
    ax.set_xticks([])
    ax.set_yticks([])
    # plot filter channel in grayscale
    plt.imshow(feature_maps[0, :, :, ix-1], cmap='gray')
    ix += 1


In [ ]:
train_images[1].shape


In [ ]:
filter = filters[:,:,:,2]
filter.shape


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming train_images[1] is your input image (32x32x3) and filter has shape (3, 3, 3)
input_image = train_images[1]


# Perform convolution
feature_map = np.zeros((input_image.shape[0] - filter.shape[0] + 1, input_image.shape[1] - filter.shape[1] + 1))

for i in range(feature_map.shape[0]):
    for j in range(feature_map.shape[1]):
        feature_map[i, j] = np.sum(input_image[i:i+filter.shape[0], j:j+filter.shape[1], :] * filter)

# Display the feature map
plt.imshow(feature_map, cmap='gray')
plt.title("Feature Map")
plt.axis('off')
plt.show()


In [ ]:
plt.imshow(feature_maps[0, :, :, 2], cmap='gray')


In [ ]:
bias = biases[2]
bias


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Assuming train_images[1] is your input image (32x32x3) and filter has shape (3, 3, 3)
input_image = train_images[1]


# Perform convolution
feature_map = np.zeros((input_image.shape[0] - filter.shape[0] + 1, input_image.shape[1] - filter.shape[1] + 1))

for i in range(feature_map.shape[0]):
    for j in range(feature_map.shape[1]):
        feature_map[i, j] = np.sum(input_image[i:i+filter.shape[0], j:j+filter.shape[1], :] * filter) + bias


# Apply ReLU activation
feature_map[feature_map < 0] = 0

# Display the feature map
plt.imshow(feature_map, cmap='gray')
plt.title("Feature Map")
plt.axis('off')
plt.show()


In [ ]:
feature_map_tmp = feature_maps[0, :, :, 2]
print(feature_map_tmp.shape)
print(feature_map_tmp[:6,:6])


In [ ]:
print(feature_map.shape)
print(feature_map[:6,:6])





<br><br><br><br><br><br>

---
#<font color='SeaGreen'>PART 3. Image classification with Multilayer perceptron</font> <img src="https://upload.wikimedia.org/wikipedia/commons/thumb/4/46/Colored_neural_network.svg/250px-Colored_neural_network.svg.png" width=150 height=150>

**Kodėl MLP silpnesnis.** Daugiasluoksnis perceptronas iškart `Flatten` paverčia 32×32×3 vaizdą 3072 skaičių vektoriumi ir neieško vietinių šablonų. CNN dalijasi filtrų svoriais erdvėje, todėl geriau tinka vaizdams. Originaliame CIFAR-10 Colab MLP gavo ~40 %, CNN ~70 %. Čia tas pats palyginimas su tf_flowers; išėjimas vėl 5 klasės.



<br><br>

---
###<font color='SeaGreen'>Step 2. Model. Multilayer perceptron</font>


In [ ]:
model = models.Sequential()
model.add(layers.Flatten(input_shape=(32, 32,3)))
model.add(layers.Dense(32, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(5))

model.compile(optimizer='adam', loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  metrics=['accuracy'])


In [ ]:
#@title Let's display the architecture of  model:

model.summary()




<br><br>

---
###<font color='SeaGreen'>Step 3. Train and evaluate the model</font>

Originalas: **15** MLP epochų. CPU: **5** epochos (pakanka pamatyti, kad perceptronas atsilieka nuo CNN).

In [ ]:
# Originaliame Colab: epochs=15. Sumažinta CPU laikui.
history = model.fit(train_images, train_labels, epochs=5, validation_data=(test_images, test_labels))


####<font color='SeaGreen'>Evaluate the model</font>


In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.1, 1])
plt.legend(loc='lower right')


In [ ]:
#@title Paprastas perceptronas silpnesnis už CNN — originaliame CIFAR-10 Colab minėta ~40 %.

test_loss, test_acc = model.evaluate(test_images,  test_labels, verbose=2)
print(test_acc)
mlp_test_acc = test_acc
print("CNN test accuracy:", cnn_test_acc)
print("MLP test accuracy:", mlp_test_acc)
